# Machine Learning for Alpha Generation & Backtesting (UK Equities)

**Goal**: build a cross-sectional long-short equity strategy on FTSE 100 stocks,
using ML-predicted relative-return rankings, and rigorously backtest it while
controlling for the usual ways quant backtests fool their authors:
look-ahead bias, survivorship bias, leakage across train/test splits, and
overlapping-window double counting.

Run `python src/data_acquisition.py` first (needs internet) to populate
`data/raw/all_prices.parquet` with real FTSE 100 data before running this
notebook, OR run `python tests/make_synthetic_data.py` to get synthetic data
for a quick pipeline smoke-test.


In [ ]:
import sys
sys.path.append('..')  # if running from notebooks/ folder

import pandas as pd
import matplotlib.pyplot as plt

from src.features import build_features_all, add_cross_sectional_rank, make_classification_target
from src.models import (
    prepare_binary_dataset, walk_forward_evaluate, train_final_model,
    generate_full_predictions, get_feature_columns,
)
from src.backtest import build_portfolio_weights, run_backtest, performance_metrics

pd.set_option('display.max_columns', 50)


## 1. Load raw data

In [ ]:
raw = pd.read_parquet('../data/raw/all_prices.parquet')
raw['Date'] = pd.to_datetime(raw['Date'])
print(raw.shape, raw['ticker'].nunique(), 'tickers')
raw.head()


## 2. Feature engineering

Every `feat_*` column uses only information available up to and including
that row's date. The target `target_fwd_return` deliberately looks forward —
that's fine for a label, never for a feature.

In [ ]:
feat = build_features_all(raw, forward_horizon=5)
feat = add_cross_sectional_rank(feat, 'feat_return_20d', 'feat_return_20d_rank')
feat = add_cross_sectional_rank(feat, 'feat_vol_20d', 'feat_vol_20d_rank')
feat = make_classification_target(feat, n_quantiles=5)

feature_cols = get_feature_columns(feat)
print(f'{len(feature_cols)} features:', feature_cols)
feat[['Date', 'ticker'] + feature_cols[:5] + ['target_fwd_return', 'target_class']].tail()


## 3. Purged, embargoed walk-forward cross-validation

We compare three model families using time-respecting CV that purges any
training row whose label window overlaps the test window, plus an embargo
buffer -- see `src/cv.py` for why this matters.

In [ ]:
binary_df = prepare_binary_dataset(feat, n_quantiles=5)
cv_results = walk_forward_evaluate(binary_df, ['logistic_regression', 'random_forest', 'lightgbm'],
                                    n_splits=5, horizon=5, embargo=5)
cv_results.groupby('model_name')['auc'].agg(['mean', 'std', 'count'])


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, g in cv_results.groupby('model_name'):
    ax.plot(g['fold'], g['auc'], marker='o', label=name)
ax.axhline(0.5, color='grey', linestyle='--', label='random baseline')
ax.set_xlabel('CV fold (walk-forward)')
ax.set_ylabel('AUC')
ax.set_title('Out-of-sample AUC by fold')
ax.legend()
plt.show()


## 4. Train final model, generate scores for backtest

In [ ]:
best_model_name = cv_results.groupby('model_name')['auc'].mean().idxmax()
print('Best model:', best_model_name)

final_model = train_final_model(binary_df, best_model_name, feature_cols)
feat['pred_score'] = generate_full_predictions(feat, final_model, feature_cols)
feat[['Date', 'ticker', 'pred_score']].dropna().tail()


## 5. Backtest

**Important**: `target_fwd_return` is a 5-day forward return recomputed every
single day, so consecutive daily rebalances would overlap by 4/5 days of the
same price moves. We only rebalance on non-overlapping dates (every 5 trading
days) to avoid double-counting -- see the comment in `src/run_pipeline.py`
for the full explanation, including the ~970% "fake" result we got before
fixing this.

In [ ]:
scored = feat.dropna(subset=['pred_score', 'target_fwd_return'])
rebalance_dates = sorted(scored['Date'].unique())[::5]
scored = scored[scored['Date'].isin(rebalance_dates)]

weighted = build_portfolio_weights(scored, score_col='pred_score', top_frac=0.15, bottom_frac=0.15)
bt_result = run_backtest(weighted, forward_return_col='target_fwd_return', holding_period=5)

metrics = performance_metrics(bt_result, periods_per_year=252/5)
for k, v in metrics.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
axes[0].plot(bt_result['Date'], bt_result['equity_curve'])
axes[0].set_title('Equity Curve')
running_max = bt_result['equity_curve'].cummax()
dd = bt_result['equity_curve'] / running_max - 1
axes[1].fill_between(bt_result['Date'], dd, 0, color='crimson', alpha=0.5)
axes[1].set_title('Drawdown')
plt.tight_layout()
plt.show()


## 6. Limitations & honest caveats

- Universe survivorship: only currently-listed FTSE 100 constituents are
  used; delisted/failed companies from the sample period are not included,
  which likely makes results look slightly better than reality.
- Transaction costs are a simplified flat bps-per-turnover estimate, not a
  real market-impact model.
- The backtest assumes we can trade exactly at the closing price used to
  compute returns, ignoring slippage and liquidity constraints for the
  smaller-cap names.
- Walk-forward AUCs in the 0.51-0.53 range are typical and expected for
  daily equity alpha signals -- a much higher AUC would be a red flag for
  leakage, not a sign of a great model.
- Past backtested performance is not indicative of future results; this
  project is for research/educational purposes only.
